# 00 Part 4 — Verify Prediction Files

**Purpose:** Confirm all 24 prediction zip files were created correctly before
moving to dataset creation.

**Checks:**
1. All 4 prediction folders exist
2. Each folder has exactly 6 zip files (24 total)
3. Each zip loads without error
4. Embedding and prediction shapes match expected row counts
5. ASIN/date keys align with main_train_keys.csv and main_val_keys.csv

**No GPU needed — CPU runtime is fine.**

## ① Setup

In [1]:
!pip install pyarrow pandas -q
print('✅ Done')

✅ Done


## ② Config

In [2]:
import os
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

# ── BRAND BUCKET — change this line for each run ───────────────────────
# Valid: women_top1_dream_pairs, women_top2_idifu, women_other,
#        men_top1_bruno_marc,    men_top2_nike,    men_other
BRAND_BUCKET = 'women_top1_dream_pairs'

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'    # shared (ASIN-unique)
PRED_DIR     = DATA_ROOT / 'predictions'
KEY_DIR      = DATA_ROOT / 'keys'
CLEAN_FILE   = DATA_DIR / 'all_products_clean.parquet'

EXPECTED_FOLDERS = [
    'txtimg/lag1',
    'txtimg/time_indipendent',
    'txt/lag1',
    'txt/time_indipendent',
]

print('✅ Config ready')
print(f'  BRAND_BUCKET    : {BRAND_BUCKET}')
print(f'  Predictions dir : {PRED_DIR}')
print(f'  Keys dir        : {KEY_DIR}')

✅ Config ready
  BRAND_BUCKET    : women_top1_dream_pairs
  Predictions dir : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/women_top1_dream_pairs/predictions
  Keys dir        : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/women_top1_dream_pairs/keys


## ③ Check All 4 Folders Exist and Have 6 Files Each

In [3]:
print('=' * 65)
print('FOLDER AND FILE COUNT CHECK')
print('=' * 65)

all_zips = {}
all_ok = True

for folder in EXPECTED_FOLDERS:
    path = PRED_DIR / folder
    exists = os.path.isdir(path)
    if not exists:
        print(f'  ❌ MISSING folder: {path}')
        all_ok = False
        continue

    zips = sorted([f for f in os.listdir(path) if f.endswith('.zip')])
    count = len(zips)
    status = '✅' if count == 6 else f'❌ expected 6, got {count}'
    print(f'  {status}  {folder}/')
    for z in zips:
        print(f'       {z}')
    all_zips[folder] = [path / z for z in zips]
    print()

total = sum(len(v) for v in all_zips.values())
print(f'Total zip files found: {total} / 24  {"✅" if total == 24 else "❌"}')
print()
if all_ok and total == 24:
    print('✅ All folders and files present — proceeding to content check.')
else:
    print('❌ Some files missing — check which Part 3 notebook needs to be re-run.')

FOLDER AND FILE COUNT CHECK
  ✅  txtimg/lag1/
       pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoch=015-val_combined_loss=0.1637_txtimg_lag1_train.zip
       pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoch=015-val_combined_loss=0.1637_txtimg_lag1_val.zip
       train_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=010-val_combined_loss=0.2550_txtimg_dim=128_proj_emb_step=.zip
       train_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=014-val_combined_loss=0.2574_txtimg_dim=256_proj_emb_step=.zip
       val_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=010-val_combined_loss=0.2550_txtimg_dim=128_proj_emb_step=.zip
       val_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=014-val_combined_loss=0.2574_txtimg_dim=256_proj_emb_step=.zip

  ✅  txtimg/time_indipendent/
       pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoch=015-val_combined_loss=0.1589_txtimg_invariant_train.zip
       pred_diff_

## ④ Load Each Zip and Check Shapes

In [4]:
print('=' * 65)
print('CONTENT CHECK — LOADING EACH ZIP')
print('=' * 65)
print()

# Load key files to get expected row counts
df_train_keys = pd.read_csv(KEY_DIR / 'main_train_keys.csv')
df_val_keys   = pd.read_csv(KEY_DIR / 'main_val_keys.csv')

print(f'Expected train rows : {len(df_train_keys):,}')
print(f'Expected val rows   : {len(df_val_keys):,}')
print()

load_errors = []

for folder, zip_paths in all_zips.items():
    print(f'--- {folder} ---')
    for zpath in zip_paths:
        fname = os.path.basename(zpath)
        try:
            with zipfile.ZipFile(zpath, 'r') as z:
                csv_files = [f for f in z.namelist() if f.endswith('.csv')]
                if not csv_files:
                    print(f'  ❌ {fname[:60]} — no CSV inside zip')
                    load_errors.append(fname)
                    continue
                df = pd.read_csv(z.open(csv_files[0]))
                rows = len(df)
                cols = len(df.columns)
                # Determine expected rows
                is_train = 'train' in fname.lower()
                is_val   = fname.lower().startswith('val')
                is_diff  = 'diff' in fname
                expected = len(df_train_keys) if is_train else len(df_val_keys) if is_val else '?'
                ok = '✅' if isinstance(expected, str) or abs(rows - expected) < 500 else '⚠️ '
                print(f'  {ok} {fname[:65]}')
                print(f'       rows={rows:,}  cols={cols}')
        except Exception as e:
            print(f'  ❌ {fname[:60]} — ERROR: {e}')
            load_errors.append(fname)
    print()

if not load_errors:
    print('✅ All zip files loaded successfully.')
else:
    print(f'❌ {len(load_errors)} file(s) had errors: {load_errors}')

CONTENT CHECK — LOADING EACH ZIP

Expected train rows : 25,194
Expected val rows   : 25,194

--- txtimg/lag1 ---
  ⚠️  pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoc
       rows=2,652  cols=132
  ✅ pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoc
       rows=2,652  cols=132
  ⚠️  train_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=01
       rows=2,873  cols=132
  ⚠️  train_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=01
       rows=2,873  cols=260
  ⚠️  val_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=010-
       rows=2,873  cols=132
  ⚠️  val_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=014-
       rows=2,873  cols=260

--- txtimg/time_indipendent ---
  ⚠️  pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoc
       rows=2,873  cols=132
  ✅ pred_diff_shoes-diff-model-txtimg_2026-06-01-embdim=768_lag1-epoc
       rows=2,873  cols=132
  ⚠️  train_pred_model-2026-06-01-embdim=None_time_indepen

## ⑤ Check ASIN/Date Key Alignment

In [5]:
print('=' * 65)
print('KEY ALIGNMENT CHECK')
print('=' * 65)
print()

train_asins = set(df_train_keys['ASIN'])
val_asins   = set(df_val_keys['ASIN'])

# Sample one zip from each folder to check alignment
for folder in EXPECTED_FOLDERS:
    path = PRED_DIR / folder
    zips = sorted([f for f in os.listdir(path) if f.endswith('.zip')])
    if not zips:
        continue

    # Pick the train level emb256 file
    sample_zip = next((z for z in zips if 'train' in z and 'dim=256' in z), zips[0])
    zpath = path / sample_zip

    try:
        with zipfile.ZipFile(zpath, 'r') as z:
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            df = pd.read_csv(z.open(csv_files[0]))

        if 'index' in df.columns:
            zip_asins = set(df['index'])
            overlap   = len(zip_asins & train_asins)
            pct       = overlap / len(zip_asins) * 100 if zip_asins else 0
            ok = '✅' if pct > 99 else '⚠️ '
            print(f'  {ok} {folder}/train: {overlap:,} / {len(zip_asins):,} ASINs valid ({pct:.1f}%)')
        else:
            print(f'  ⚠️  {folder}: index column not found in {sample_zip}')
    except Exception as e:
        print(f'  ❌ {folder}: {e}')

print()

KEY ALIGNMENT CHECK

  ✅ txtimg/lag1/train: 221 / 221 ASINs valid (100.0%)
  ✅ txtimg/time_indipendent/train: 221 / 221 ASINs valid (100.0%)
  ✅ txt/lag1/train: 221 / 221 ASINs valid (100.0%)
  ✅ txt/time_indipendent/train: 221 / 221 ASINs valid (100.0%)



In [6]:
# ── Quick diagnostic ──────────────────────────────────────────
import zipfile

# Pick one failing zip (lag1 train dim=256)
folder = 'txtimg/lag1'
zips = sorted(os.listdir(PRED_DIR / folder))
sample = next((z for z in zips if 'train' in z and 'dim=256' in z), zips[0])
zpath  = PRED_DIR / folder / sample

with zipfile.ZipFile(zpath) as z:
    df_zip = pd.read_csv(z.open(z.namelist()[0]))

df_keys = pd.read_csv(KEY_DIR / 'main_train_keys.csv')

zip_asins   = set(df_zip['index'])
train_asins = set(df_keys['ASIN'])

print(f'ZIP file selected : {sample}')
print(f'ZIP rows          : {len(df_zip):,}')
print(f'ZIP unique ASINs  : {len(zip_asins):,}')
print(f'ZIP dtype index   : {df_zip["index"].dtype}')
print(f'ZIP first 5 ASINs : {list(df_zip["index"].unique()[:5])}')
print()
print(f'Keys rows         : {len(df_keys):,}')
print(f'Keys unique ASINs : {len(train_asins):,}')
print(f'Keys dtype ASIN   : {df_keys["ASIN"].dtype}')
print(f'Keys first 5 ASINs: {list(df_keys["ASIN"].unique()[:5])}')
print()
print(f'Overlap           : {len(zip_asins & train_asins):,}')
print(f'In ZIP not keys   : {len(zip_asins - train_asins):,}')
print(f'In keys not ZIP   : {len(train_asins - zip_asins):,}')

ZIP file selected : train_pred_model-2026-06-01-embdim=None_lag1_txtimg_mod4-epoch=014-val_combined_loss=0.2574_txtimg_dim=256_proj_emb_step=.zip
ZIP rows          : 2,873
ZIP unique ASINs  : 221
ZIP dtype index   : object
ZIP first 5 ASINs : ['B014VGRQUK', 'B016E4SV8S', 'B016E4T8BM', 'B016E4TMLI', 'B016E4TMNQ']

Keys rows         : 25,194
Keys unique ASINs : 221
Keys dtype ASIN   : object
Keys first 5 ASINs: ['B014VGRQUK', 'B016E4SV8S', 'B016E4T8BM', 'B016E4TMLI', 'B016E4TMNQ']

Overlap           : 221
In ZIP not keys   : 0
In keys not ZIP   : 0


## ⑥ Final Verdict

In [7]:
print('=' * 65)
print('FINAL VERIFICATION SUMMARY')
print('=' * 65)

total_found = sum(len(v) for v in all_zips.values())

print(f'''
Folders checked : {len(EXPECTED_FOLDERS)}
Zip files found : {total_found} / 24
Load errors     : {len(load_errors)}
''')

if total_found == 24 and len(load_errors) == 0:
    print('✅ ALL CHECKS PASSED')
    print()
    print('Next steps:')
    print('  1. Run 00_part5_create_image_parquet.ipynb (CPU, ~15 min)')
    print('  2. Run 01_1_create_dataset_txt_img.ipynb   (local machine)')
    print('  3. Run 01_2_create_dataset_txt.ipynb       (local machine)')
else:
    print('❌ SOME CHECKS FAILED — review output above')
    if total_found < 24:
        missing = 24 - total_found
        print(f'   {missing} zip file(s) missing — re-run the corresponding Part 3 notebook')
    if load_errors:
        print(f'   {len(load_errors)} file(s) could not be loaded — may be corrupted')

FINAL VERIFICATION SUMMARY

Folders checked : 4
Zip files found : 24 / 24
Load errors     : 0

✅ ALL CHECKS PASSED

Next steps:
  1. Run 00_part5_create_image_parquet.ipynb (CPU, ~15 min)
  2. Run 01_1_create_dataset_txt_img.ipynb   (local machine)
  3. Run 01_2_create_dataset_txt.ipynb       (local machine)
